In [1]:
from google.colab import files
train_df = files.upload()

Saving UNSW_NB15_training-set.parquet to UNSW_NB15_training-set.parquet


In [2]:
from google.colab import files
test_df = files.upload()

Saving UNSW_NB15_testing-set.parquet to UNSW_NB15_testing-set.parquet


In [4]:
import pandas as pd
train_df = pd.read_parquet("UNSW_NB15_training-set.parquet")
test_df = pd.read_parquet("UNSW_NB15_testing-set.parquet")

In [5]:
!pip install xgboost -q

In [6]:
import numpy as np

from xgboost import XGBClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [7]:
X_train = train_df.drop(
    columns=["label", "attack_cat"]
)

y_train = train_df["label"]

X_test = test_df.drop(
    columns=["label", "attack_cat"]
)

y_test = test_df["label"]

In [9]:
import pandas as pd

for col in ["proto", "service", "state"]:
    # Convert training column to categorical type, defining its categories
    X_train[col] = X_train[col].astype('category')

    # Get the categories from the training set
    train_categories = X_train[col].cat.categories

    # Convert the test column to categorical type using only the categories observed in the training set
    # Any value in X_test[col] not in train_categories will become NaN (which will be -1 after .cat.codes)
    X_test[col] = X_test[col].astype(pd.CategoricalDtype(categories=train_categories))

    # Now, convert these categorical columns to their numerical codes
    # NaN values (unseen categories in X_test) will be represented as -1
    X_train[col] = X_train[col].cat.codes
    X_test[col] = X_test[col].cat.codes

In [11]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

In [12]:
xgb.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [13]:
y_pred_xgb = xgb.predict(
    X_test
)

In [15]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb.feature_importances_
})

importance.sort_values(
    by="importance",
    ascending=False
).head(20)

,feature,importance
10,dload,0.425789
23,ackdat,0.086421
9,sload,0.074693
11,sloss,0.047749
29,ct_dst_sport_ltm,0.047562
4,spkts,0.043538
1,proto,0.038713
2,service,0.025553
33,is_sm_ips_ports,0.022521
12,dloss,0.020096


In [16]:
print(
    "Accuracy:",
    accuracy_score(y_test, y_pred_xgb)
)

print(
    "Precision:",
    precision_score(y_test, y_pred_xgb)
)

print(
    "Recall:",
    recall_score(y_test, y_pred_xgb)
)

print(
    "F1:",
    f1_score(y_test, y_pred_xgb)
)

print(
    confusion_matrix(
        y_test,
        y_pred_xgb
    )
)

print(
    classification_report(
        y_test,
        y_pred_xgb
    )
)

Accuracy: 0.7982801340912403
Precision: 0.7507157321416102
Recall: 0.9486455483984824
F1: 0.8381538941295704
[[22720 14280]
 [ 2328 43004]]
              precision    recall  f1-score   support

           0       0.91      0.61      0.73     37000
           1       0.75      0.95      0.84     45332

    accuracy                           0.80     82332
   macro avg       0.83      0.78      0.79     82332
weighted avg       0.82      0.80      0.79     82332

